# *5G deployment on already provisioned infrastructure*

### **Description**:
- #### Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
- #### OAI gNB with USRP B210

### Install Post-5G orchestrator:

In [ ]:
! pip install ../../ --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple

## Load input models:

- #### Meta Model:

In [1]:
from metamodel.models.metamodel import MetaModel

metamodel = MetaModel.from_yaml(
"""
rus:
  b210:
    type: usrp
    model: b210
    rxGainDB: 114

ran:
  components:
    gnb:
      type: gnb
      config:
        radioConfiguration:
          nrBand: 78
          pointAFrequencyArfcn: 639996
          channelBandwidthMhz: 20
          ssbArfcn: 640704
        ru: b210
"""
)

- #### Deployment Model:

In [2]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
nodes:
  ptb-04015558.inria.fr: # node's actual hostname
    is_provisioned: true
    provisioner: null
    ssh_info:
      hostname: 138.96.193.106

rus:
  b210:
    is_provisioned: true
    provisioner: null

core:
  implementation: open5gs
  database_params:
    image:
      repository: "docker.io/mongo"
      image_tag: "7.0" # this is the mongodb version compatible with the machine's kernel version

ran:
  implementation: oai
  components:
    gnb:
      type: gnb
      advanced_config:
        # reference: https://github.com/duranta-project/openairinterface5g/blob/develop/ci-scripts/conf_files/gnb.sa.band78.51prb.usrpb200.conf
        initialDLBWPlocationAndBandwidth: 13750
        initialDLBWPcontrolResourceSetZero: 12
        initialULBWPlocationAndBandwidth: 13750
        ssb_perRACH_OccasionAndCB_PreamblesPerSSB: 15
        pusch_TargetSNRx10: 200
        pucch_TargetSNRx10: 200
        prach_dtx_threshold: 120
        pucch0_dtx_threshold: 100
"""
)

## **1.**   Kubernetes Deployement:

In [3]:
from orchestrator.deployment import K8SManagement

k8s = K8SManagement(metamodel=metamodel, deployment_model=deployment_model)

In [4]:
await k8s.deploy()


PLAY [ptb-04015558.inria.fr] ***************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'ptb-04015558.inria.fr'] ***
[WARNING]: Host 'ptb-04015558.inria.fr' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [ptb-04015558.inria.fr]

TASK [k8s/prepare : Install base packages required for Kubernetes on 'ptb-04015558.inria.fr'] ***
changed: [ptb-04015558.inria.fr]

TASK [k8s/prepare : Ensure apt keyring directory exists on 'ptb-04015558.inria.fr'] ***
ok: [ptb-04015558.inria.fr]

TASK [k8s/prepare : Add Kubernetes apt signing key on 'ptb-04015558.inria.fr'] ***
changed: [ptb-04015558.inria.fr]

TASK [k8s/prepare : Convert Kubernetes apt key to keyring format on 'ptb-04015558.inria.fr'] ***
changed

## **2.**   5G Core Deployement:

In [6]:
from orchestrator.deployment import CoreManagement

core = CoreManagement(k8s)

In [7]:
await core.deploy()


PLAY [ptb-04015558.inria.fr] ***************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-amf' chart] ******

PLAY [ptb-04015558.inria.fr] ***************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-database' chart] ***

PLAY [ptb-04015558.inria.fr] ***************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-bsf' chart] ******

PLAY [ptb-04015558.inria.fr] ***************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-ausf' chart] *****

PLAY [ptb-04015558.inria.fr] ***************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-smf' chart] ******

PLAY [ptb-04015558.inria.fr] ***************************************************

PLAY [ptb-04015558.inria.fr] ***************************************************

TASK [helm/pr

### Check deployed pods with `kubectl`:

In [8]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(core)

print((await prov.run_node_command(command=["kubectl", "get", "pods", "-n", "open5gs-core"], become=True, quiet=True))["ptb-04015558.inria.fr"])

NAME                            READY   STATUS    RESTARTS   AGE
open5gs-amf-7b6dcfbfdf-gnjsb    1/1     Running   0          2m
open5gs-ausf-5778d7548b-jgxg4   1/1     Running   0          106s
open5gs-bsf-75cf9d7fb4-chw8b    1/1     Running   0          17s
open5gs-database-0              1/1     Running   0          3m15s
open5gs-nrf-65d98494d7-rckvd    1/1     Running   0          2m27s
open5gs-nssf-84cb8d64f7-5tp6h   1/1     Running   0          32s
open5gs-pcf-6f444b66c5-r55gg    1/1     Running   0          76s
open5gs-smf-5c7c8d4c4c-5hntb    1/1     Running   0          91s
open5gs-udm-886ffcdb8-6n75c     1/1     Running   0          61s
open5gs-udr-77bbf9559-d2jwv     1/1     Running   0          46s
open5gs-upf-55c6894777-pfclw    1/1     Running   0          76s


### Check if USRP B210 is plugged in:

In [9]:
print((await prov.run_node_command(command=["lsusb"], become=True, quiet=True))["ptb-04015558.inria.fr"])

Bus 001 Device 001: ID 1d6b:0002 Linux Foundation 2.0 root hub
Bus 001 Device 002: ID 0a5c:5832 Broadcom Corp. 5880
Bus 001 Device 003: ID 0c45:6717 Microdia Integrated_Webcam_HD
Bus 002 Device 001: ID 1d6b:0003 Linux Foundation 3.0 root hub
Bus 002 Device 002: ID 2500:0020 Ettus Research LLC USRP B210


## **3.**   RAN deployement:

In [10]:
from orchestrator.deployment import RANManagement

ran = RANManagement(core)

In [11]:
await ran.deploy()


PLAY [ptb-04015558.inria.fr] ***************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-gnb' chart] **********
[WARNING]: Host 'ptb-04015558.inria.fr' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [ptb-04015558.inria.fr]

TASK [helm/prepare : Copy local 'oai-gnb' Helm chart directory for 'oai-gnb'] ***
ok: [ptb-04015558.inria.fr]

TASK [helm/prepare : Render 'oai-gnb' Helm Chart.yaml file from Jinja template] ***
ok: [ptb-04015558.inria.fr]

TASK [helm/prepare : Render templated Helm files for 'oai-gnb'] ****************
changed: [ptb-04015558.inria.fr] => (item=values.yaml)
changed: [ptb-04015558.inria.fr] => (item=config.yaml)

PLAY RECAP **********************************************

In [17]:
print((await ran.get_logs(quiet=True))["gnb"])

/proc/sys/kernel/core_pattern=|/usr/share/apport/apport -p%p -s%s -c%c -d%d -P%P -u%u -g%g -F%F -- %E
No configuration file /opt/oai-gnb/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

Active_gNBs:
  - gnb
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  ##### Identification parameters:
  - gNB_ID: 0x1

    gNB_name: gnb

    tracking_area_code: 1

    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1

    nr_cellid: 1

    ##### Physical parameters:
    min_rxtxtime: 6

    servingCellConfigCommon:
      #spCellConfigCommon
      - physCellId: 0
        #  downlinkConfigCommon
        #frequencyInfoDL
        # this is 3300.60 MHz + 53*12*30e-3 MHz = 3319.68
        absoluteFrequencySSB: 640704
        # this is 3300.60 MHz
        dl_absoluteFrequencyPointA: 639996
        #scs-SpecificCarrierList
        dl_offstToCarrier: 0
        # subcarrierSpacing
        # 0=kHz15, 1

### Save final models:

In [13]:
ran.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

(MetaModel(core=Core(components={'amf': AMFComponent(category=<Category5G.CORE: 'core'>, type=<CoreComponentType.AMF: 'amf'>, config=AMFConfig(sbi=SBI(interfaceName='eth0', ipv4EndPointAddress=IPv4Address('0.0.0.0'), ipv6EndPointAddress=None, port=80, protocol='TCP', scheme='http', apiVersion='v1', tls=TLS(enabled=False, certificate=None, privateKey=None)), nrfUri=None, n2=NGAP(interfaceName='eth0', ipv4EndPointAddress=IPv4Address('0.0.0.0'), ipv6EndPointAddress=None, port=38412, protocol='SCTP'), guamiList=[GUAMI(plmnId=PLMN(mcc=1, mnc=1, mncLength=2, plmnId='00101'), amfRegionId=2, amfSetId=1, amfPointer=0, amfId=131136)], taiList=[TAI(plmnId=PLMN(mcc=1, mnc=1, mncLength=2, plmnId='00101'), tac=1)], plmnList=[PLMNSupportItem(plmnId=PLMN(mcc=1, mnc=1, mncLength=2, plmnId='00101'), snssaiList=['internet'])], securityAlgorithmConfiguration=SecurityAlgorithmConfiguration(integrityOrder=[<IntegrityAlgorithm.NIA2: 'nia2'>, <IntegrityAlgorithm.NIA1: 'nia1'>, <IntegrityAlgorithm.NIA0: 'nia0'

## Cleanup:

- #### RAN cleanup:

In [14]:
# from orchestrator.deployment import RANManagement

# ran = RANManagement(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

# await ran.remove(force=True)

- #### 5G core cleanup:

In [15]:
# from orchestrator.deployment import CoreManagement

# core = CoreManagement(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

# await core.remove(force=True)

- #### Kubernetes cleanup:

In [16]:
# from orchestrator.deployment import K8SManagement

# k8s = K8SManagement(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

# await k8s.remove(force=True)